In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


### Get all programs

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

#--------- chose a disease --------------------
DISEASE_ID = 'ACC'
DISEASE_ID = 'PAAD'

In [6]:
verbose=False
force=False

method='deseq2'
imax_tumor=250
imax_normal=50

gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

exclude_prog_list=['CCLE']

dfn_tumor, dfn_normal, df_gtex, df_ana = gdc.get_all_data_from_disease(disease_id=DISEASE_ID, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

df_ana



>> dfn_tumor (60616, 460)
>> dfn_normal (60616, 103)


,prog_id,psi_id,disease_id,primary_site
0,CPTAC,PAAD,PAAD,Pancreas
1,CPTAC,PAAD_GDC,PAAD,Pancreas
2,TCGA,TCGA-PAAD,PAAD,Pancreas


### Clusterization

In [7]:
group = 'Tumor'

verbose=False
force=False

disease_id=DISEASE_ID
imax_tumor=250
imax_normal=50
exclude_prog_list=['CCLE']
n_components = 10
min_clusters = 6; max_clusters = 12
n_umap_neighbors=5; min_umap_dist=0.2; umap_metric="euclidean"
method_hca="ward"; hca_criterion="maxclust"
LFC_cutoff=1; FDR_cutoff=0.05
perc_min_samples=0.25; top_n=10_000

if group == 'Tumor':
    df = dfn_tumor
    n_clusters = 10
else:
    df = dfn_normal
    n_clusters = 3


df_cluster, df_sel, df_cpm, df_pca, df_umap = gdc.cluster_expression_data_group(df, group=group, n_clusters=n_clusters, 
                                                n_components=n_components, min_clusters=min_clusters, max_clusters=max_clusters,
                                                n_umap_neighbors=n_umap_neighbors, min_umap_dist=min_umap_dist, umap_metric=umap_metric,
                                                method_hca=method_hca, hca_criterion=hca_criterion,
                                                LFC_cutoff=LFC_cutoff, FDR_cutoff=FDR_cutoff,
                                                perc_min_samples=perc_min_samples, top_n=top_n,
                                                force=force, verbose=verbose)

In [8]:
dfc_log, dfc_log_nor = gdc.prepare_log_expression(df_cpm=df_cpm, df_sel=df_sel, dfn_normal=dfn_normal, top_n=top_n)


dfc_log.head(3)

10000


,1,2,3,4,5,6,7,8,9,10,...,448,449,450,451,452,453,454,455,456,457
ENSG00000270641,14.479,13.720,14.420,13.992,14.229,12.156,13.862,13.964,13.765,13.668,...,9.388,0.107,9.473,0.350,0.339,9.473,5.700,0.359,8.507,0.339
ENSG00000262619,10.816,10.877,12.968,12.899,14.828,9.721,12.547,10.455,9.238,13.277,...,0.129,0.157,0.833,1.068,1.927,0.055,0.734,0.359,0.522,0.756
ENSG00000281383,3.418,0.000,10.809,0.000,11.366,5.782,3.113,0.000,0.000,0.759,...,0.248,0.206,0.135,0.242,0.885,0.301,0.000,0.249,0.000,0.398


In [9]:
dfc_log_nor.head(3)

,458,459,460,461,462,463,464,465,466,467,...,548,549,550,551,552,553,554,555,556,557
geneid,,,,,,,,,,,,,,,,,,,,,
ENSG00000000971,4.040,2.152,1.815,2.213,2.661,0.765,0.000,1.641,2.880,3.745,...,3.176,0.805,3.168,2.644,0.000,2.488,1.079,2.808,3.059,2.459
ENSG00000001084,1.882,2.542,0.000,1.495,2.069,1.631,0.861,1.641,2.471,2.184,...,1.770,0.000,1.873,0.000,1.393,1.956,3.950,1.585,1.874,0.807
ENSG00000001461,2.124,3.316,2.737,2.548,3.080,1.631,0.861,2.571,4.674,2.184,...,2.327,2.639,1.584,0.728,0.859,1.956,1.846,1.875,3.544,3.999


### Merge tumor + control without combat correction

In [10]:
dfn = pd.merge(dfc_log, dfc_log_nor, left_index=True, right_index=True, how='inner')
print(dfn.shape)

dfn.head(3)

(10000, 557)


,1,2,3,4,5,6,7,8,9,10,...,548,549,550,551,552,553,554,555,556,557
ENSG00000270641,14.479,13.720,14.420,13.992,14.229,12.156,13.862,13.964,13.765,13.668,...,3.042,3.204,5.332,2.484,3.808,4.869,13.583,4.755,3.841,14.448
ENSG00000262619,10.816,10.877,12.968,12.899,14.828,9.721,12.547,10.455,9.238,13.277,...,1.383,1.696,2.320,0.728,0.859,2.488,2.714,4.393,4.029,2.643
ENSG00000281383,3.418,0.000,10.809,0.000,11.366,5.782,3.113,0.000,0.000,0.759,...,0.000,0.000,0.000,0.000,0.000,0.000,1.312,0.000,0.737,0.000


In [11]:
df_metadata = gdc.open_metadata(verbose=True)
df_metadata.tail(3)

Table opened ((557, 3)) at '/home/flavio/uv/perturb_agent/data/multi_progs/lfc/metadata_tumor_and_normal.tsv'


,condition,dataset,condition_numeric
554,normal,CPTAC_GDC,0.0
555,normal,CPTAC_GDC,0.0
556,normal,CPTAC_GDC,0.0


In [12]:
df_metadata.index.dtype

dtype('int64')

In [13]:
df_combat = gdc.open_combat_input_log_cpm(verbose=True)
df_combat.head(3)

Table opened ((10000, 560)) at '/home/flavio/uv/perturb_agent/data/multi_progs/lfc/combat_log_exp_tumor_and_normal.tsv'


,geneid,symbol,biotype,1,2,3,4,5,6,7,...,548,549,550,551,552,553,554,555,556,557
0,ENSG00000000971,CFH,protein_coding,2.246,0.482,2.755,2.204,4.640,0.437,0.524,...,2.887,0.824,2.881,2.424,0.124,2.288,1.062,2.567,2.785,2.263
1,ENSG00000001084,GCLC,protein_coding,2.910,2.282,2.449,2.903,5.244,1.058,3.153,...,1.636,-0.035,1.732,-0.035,1.280,1.811,3.692,1.461,1.734,0.727
2,ENSG00000001461,NIPAL3,protein_coding,4.104,4.274,2.748,0.284,2.603,0.484,2.438,...,2.198,2.476,1.537,0.776,0.892,1.868,1.770,1.796,3.281,3.686


In [14]:
df_combat.columns

Index([ 'geneid',  'symbol', 'biotype',         1,         2,         3,         4,         5,
               6,         7,
       ...
             548,       549,       550,       551,       552,       553,       554,       555,
             556,       557],
      dtype='object', length=560)

### Clusters

In [15]:
df_hca = gdc.df_hca
df_hca

,sample,cluster
0,1,7
1,2,1
2,3,7
3,4,1
4,5,6
...,...,...
452,453,5
453,454,5
454,455,5
455,456,5


In [16]:
df_hca.cluster.max()

10

In [17]:
force=False
verbose=True

nclu=10

cols = df_hca[df_hca.cluster == nclu]['sample']
gene_cols = ["geneid", "symbol", "biotype"]
all_cols = gene_cols + list(cols)

dfn_combat_tum = df_combat[all_cols].copy()
dfn_combat_tum


,geneid,symbol,biotype,48,111,120,131,262,324,333,344
0,ENSG00000000971,CFH,protein_coding,4.308,6.120,-0.203,6.613,3.330,4.627,0.102,4.980
1,ENSG00000001084,GCLC,protein_coding,3.127,4.961,5.789,7.467,2.878,4.522,5.264,6.768
2,ENSG00000001461,NIPAL3,protein_coding,7.896,6.712,6.620,5.741,6.445,5.516,5.444,4.755
3,ENSG00000001617,SEMA3F,protein_coding,7.888,4.245,4.067,4.397,6.512,3.594,3.451,3.716
4,ENSG00000001626,CFTR,protein_coding,6.838,4.306,-0.325,-0.325,6.393,4.311,0.505,0.505
...,...,...,...,...,...,...,...,...,...,...,...
9995,ENSG00000288597,AC234782.2,lncRNA,5.494,4.855,6.865,4.250,5.480,4.860,6.810,4.274
9996,ENSG00000288598,AL354833.1,lncRNA,6.737,2.154,-2.621,2.291,6.742,4.106,1.360,4.185
9997,ENSG00000288611,NPBWR1,protein_coding,-0.155,6.181,6.779,6.075,0.181,6.217,6.787,6.116
9998,ENSG00000288612,AL133351.4,lncRNA,-0.284,-0.284,1.810,-0.284,0.228,0.228,2.132,0.228


In [18]:
df_tumor = dfn_combat_tum
df_tumor.head(3)

,geneid,symbol,biotype,48,111,120,131,262,324,333,344
0,ENSG00000000971,CFH,protein_coding,4.308,6.120,-0.203,6.613,3.330,4.627,0.102,4.980
1,ENSG00000001084,GCLC,protein_coding,3.127,4.961,5.789,7.467,2.878,4.522,5.264,6.768
2,ENSG00000001461,NIPAL3,protein_coding,7.896,6.712,6.620,5.741,6.445,5.516,5.444,4.755


In [19]:
cols_normal = df_metadata[df_metadata.condition == 'normal'].index.to_list()

In [20]:

df_normal = df_combat[["geneid", "symbol", "biotype"] + cols_normal]
df_normal.head(3)

,geneid,symbol,biotype,457,458,459,460,461,462,463,...,547,548,549,550,551,552,553,554,555,556
0,ENSG00000000971,CFH,protein_coding,2.474,4.674,2.379,1.969,2.453,2.998,0.692,...,2.445,2.887,0.824,2.881,2.424,0.124,2.288,1.062,2.567,2.785
1,ENSG00000001084,GCLC,protein_coding,4.508,1.991,2.685,0.010,1.583,2.187,1.726,...,1.844,1.636,-0.035,1.732,-0.035,1.280,1.811,3.692,1.461,1.734
2,ENSG00000001461,NIPAL3,protein_coding,5.220,2.300,3.651,2.995,2.781,3.384,1.741,...,1.899,2.198,2.476,1.537,0.776,0.892,1.868,1.770,1.796,3.281


In [21]:
df_exp, df_annotation = gdc.prepare_expression_matrix(
                df_tumor=df_tumor,
                df_normal=df_normal,
            )

df_exp.head(3)

,48,111,120,131,262,324,333,344,457,458,...,547,548,549,550,551,552,553,554,555,556
geneid,,,,,,,,,,,,,,,,,,,,,
ENSG00000000971,4.308,6.120,-0.203,6.613,3.330,4.627,0.102,4.980,2.474,4.674,...,2.445,2.887,0.824,2.881,2.424,0.124,2.288,1.062,2.567,2.785
ENSG00000001084,3.127,4.961,5.789,7.467,2.878,4.522,5.264,6.768,4.508,1.991,...,1.844,1.636,-0.035,1.732,-0.035,1.280,1.811,3.692,1.461,1.734
ENSG00000001461,7.896,6.712,6.620,5.741,6.445,5.516,5.444,4.755,5.220,2.300,...,1.899,2.198,2.476,1.537,0.776,0.892,1.868,1.770,1.796,3.281


In [22]:
df_annotation.head(3)

,geneid,symbol,biotype
0,ENSG00000000971,CFH,protein_coding
1,ENSG00000001084,GCLC,protein_coding
2,ENSG00000001461,NIPAL3,protein_coding


In [67]:
from inmoose.limma import lmFit, eBayes, topTable, makeContrasts, contrasts_fit

tumor_samples = [
    col for col in df_tumor.columns
    if col not in gdc.ANNOT_COLS
]

normal_samples = [
    col for col in df_normal.columns
    if col not in gdc.ANNOT_COLS
]

metadata = pd.DataFrame(
    {
        "group": (
            ["Tumor"] * len(tumor_samples) + 
            ["Normal"] * len(normal_samples)
        )
    },
    index=tumor_samples + normal_samples,
)

metadata = metadata.loc[df_exp.columns].copy()

# Intercept + binary tumor coefficient
# Normal is the reference condition
design = pd.DataFrame(
    {
        "Intercept": 1.0,
        "Tumor_vs_Normal": (
            metadata["group"].eq("Tumor").astype(float)
        ),
    },
    index=metadata.index,
)

assert df_exp.columns.equals(design.index)

fit = lmFit(df_exp, design)

design

,Intercept,Tumor_vs_Normal
48,1.0,1.0
111,1.0,1.0
120,1.0,1.0
131,1.0,1.0
262,1.0,1.0
...,...,...
552,1.0,0.0
553,1.0,0.0
554,1.0,0.0
555,1.0,0.0


In [68]:
# InMoose supports trend=True for logCPM-like data and calculates moderated statistics through empirical-Bayes variance shrinkage.
coefficient_names = ["Intercept", "Tumor_vs_Normal"]

fit.coefficients.columns = coefficient_names
fit.stdev_unscaled.columns = coefficient_names


fit_bayes = eBayes(
    fit,
    trend=True,
    robust=False,
)

In [69]:
print(fit.coefficients.columns)
print(fit_bayes.coefficients.columns)

Index(['Intercept', 'Tumor_vs_Normal'], dtype='object')
Index(['Intercept', 'Tumor_vs_Normal'], dtype='object')


In [70]:
print("coefficients:", fit_bayes.coefficients.columns.tolist())
print("stdev_unscaled:", fit_bayes.stdev_unscaled.columns.tolist())
print("t:", fit_bayes.t.columns.tolist())
print("p_value:", fit_bayes.p_value.columns.tolist())
print("lods:", fit_bayes.lods.columns.tolist())

coefficients: ['Intercept', 'Tumor_vs_Normal']
stdev_unscaled: ['Intercept', 'Tumor_vs_Normal']
t: ['Intercept', 'Tumor_vs_Normal']
p_value: ['Intercept', 'Tumor_vs_Normal']
lods: ['Intercept', 'Tumor_vs_Normal']


In [71]:
gene_annotations = df_annotation.copy()

gene_annotations.index = gene_annotations["geneid"].astype(str)
df_exp.index = df_exp.index.astype(str)

gene_annotations = gene_annotations.loc[df_exp.index]
gene_annotations

,geneid,symbol,biotype
geneid,,,
ENSG00000000971,ENSG00000000971,CFH,protein_coding
ENSG00000001084,ENSG00000001084,GCLC,protein_coding
ENSG00000001461,ENSG00000001461,NIPAL3,protein_coding
ENSG00000001617,ENSG00000001617,SEMA3F,protein_coding
ENSG00000001626,ENSG00000001626,CFTR,protein_coding
...,...,...,...
ENSG00000288597,ENSG00000288597,AC234782.2,lncRNA
ENSG00000288598,ENSG00000288598,AL354833.1,lncRNA
ENSG00000288611,ENSG00000288611,NPBWR1,protein_coding


In [72]:
print(df_exp.index.equals(fit_bayes.coefficients.index))
print(df_exp.index.equals(fit_bayes.t.index))
print(df_exp.index.equals(gene_annotations.index))

True
True
True


In [73]:
assert len(df_annotation) == len(df_exp)

In [101]:
results = topTable(
    fit_bayes,
    coef="Tumor_vs_Normal",
    number=df_exp.shape[0],
    # genelist=gene_annotations,
    adjust_method="fdr_bh",
    sort_by="P",
)

df_lfc_ori = pd.DataFrame(
    data=results.to_numpy(copy=True),
    index=results.index.copy(),
    columns=results.columns.copy(),
)

df_lfc_ori = df_lfc_ori.merge(
    df_annotation,
    on="geneid",
    how="left",
    validate="one_to_one",
)

cols = ['geneid', 'lfc', 'lfcSE', 'ave_expr', 'stat', 'pvalue', 'fdr', 'B', 'symbol', 'biotype']
df_lfc_ori.columns = cols

cols = ['geneid', 'symbol', 'biotype', 'lfc', 'abs_lfc', 'lfcSE', 'pvalue', 'fdr', 'ave_expr', 'stat', 'B']
df_lfc_ori['abs_lfc'] = df_lfc_ori['lfc'].abs()
df_lfc_ori = df_lfc_ori[cols]

df_lfc_ori

,geneid,symbol,biotype,lfc,abs_lfc,lfcSE,pvalue,fdr,ave_expr,stat,B
0,ENSG00000271207,MTCO1P22,processed_pseudogene,6.799e+00,6.799e+00,0.367,1.169e-87,1.159e-83,0.537,5.904e+01,177.118
1,ENSG00000210112,MT-TM,Mt_tRNA,8.065e+00,8.065e+00,0.367,2.318e-87,1.159e-83,0.712,5.868e+01,176.592
2,ENSG00000240409,MTATP8P1,unprocessed_pseudogene,8.426e+00,8.426e+00,0.367,1.990e-83,6.634e-80,0.693,5.406e+01,169.504
3,ENSG00000285437,POLR2J3,protein_coding,6.585e+00,6.585e+00,0.367,1.126e-75,2.781e-72,0.635,4.594e+01,154.854
4,ENSG00000262902,MTCO1P40,processed_pseudogene,6.833e+00,6.833e+00,0.367,1.390e-75,2.781e-72,0.663,4.585e+01,154.676
...,...,...,...,...,...,...,...,...,...,...,...
9995,ENSG00000165168,CYBB,protein_coding,9.012e-03,9.012e-03,0.367,9.879e-01,9.883e-01,2.442,1.522e-02,-6.900
9996,ENSG00000177301,KCNA2,protein_coding,-1.166e-02,1.166e-02,0.367,9.918e-01,9.921e-01,5.837,-1.031e-02,-6.900
9997,ENSG00000134874,DZIP1,protein_coding,-5.572e-03,5.572e-03,0.367,9.927e-01,9.929e-01,4.194,-9.161e-03,-6.900
9998,ENSG00000160055,TMEM234,protein_coding,-5.034e-03,5.034e-03,0.367,9.944e-01,9.945e-01,4.192,-7.059e-03,-6.900


In [102]:
";".join(df_lfc_ori.symbol)

'MTCO1P22;MT-TM;MTATP8P1;POLR2J3;MTCO1P40;AC005041.5;AC135507.1;CYB561D2;MTND6P22;CFAP73;AC006017.1;AC084125.1;AC105328.1;AC090607.1;UBE2Q1-AS1;MTCO2P2;AP000781.1;NR2C2AP;AP001160.3;AC013489.3;AC007193.3;HCRTR1;AP003307.1;DLX6-AS1;AL021707.5;AC091153.4;STRADB;AC005306.1;RSPH9;MIR1282;AC005943.2;AP003068.1;AC005045.2;ARAP1-AS1;ASH1L-AS1;BRICD5;TMEM18-DT;AP002433.1;AL031587.2;AC002401.1;NUCB1-AS1;AL049569.1;AC073263.1;RNA5SP216;AC010542.5;AL390208.1;LINC00337;AL132639.3;AC020558.1;AL513218.1;C14orf178;AC009065.7;MTCO2P12;AL358937.1;MROH8;AC127024.6;MTND6P4;MCCC1-AS1;U47924.1;AL929472.3;AP001458.1;AL078612.1;AL603839.1;AL110504.1;AC005262.2;AC013472.2;AL354920.1;AL353708.3;AL353689.2;AL355310.2;AC092118.2;AC008915.2;AC009113.2;AC231981.1;SLC6A1;AC020934.2;AF129075.2;AL138724.1;AC009137.1;MT-ATP8;U47924.2;MUC2;MRAP;COA6-AS1;AC006547.3;AC009065.1;MORF4L2-AS1;AC022382.1;AL929472.2;AC023055.1;AC022201.2;AC073389.2;AC025283.2;FBXW7-AS1;ZNF649-AS1;AL513550.1;C1QTNF7-AS1;AC145207.8;GDF9;EGILA;AC

In [87]:
df_lfc_ori.columns 

Index(['geneid', 'log2FoldChange', 'lfcSE', 'AveExpr', 'stat', 'pvalue', 'adj_pvalue', 'B',
       'symbol', 'biotype'],
      dtype='object')

In [ ]:
method='deseq2'
force=True
verbose=True

df_lfc, df_lfc_ori, degs_txt, msg = gdc.calc_degs(
                                        prog_id = PROG_ID, 
                                        psi_id = PSI_ID,
                                        ncluster = nclu,
                                        df_tumor = dfn_combat_tum,
                                        df_normal = dfn_combat_norm, 
                                        df_gtex_ctrl = pd.DataFrame(),
                                        root_lfc= gdc.root_mprog_lfc,
                                        root_src = gdc.root_src,
                                        run_conda = True,
                                        lfc_cutoff = 1.0,
                                        fdr_cutoff = 0.05,
                                        method = method,
                                        force = force,
                                        verbose = verbose,
                                        )

In [ ]:
df_lfc